# Distances Between Distributions

**Audience:** students who know KL and Jensen-Shannon divergence.

**What you will build:** three genuine metrics on distributions --- Hellinger,
Wasserstein and total variation --- with concrete Bernoulli and continuous
examples, plus a comparison that shows why Wasserstein uses geometry.

## Learning objectives

1. compute Hellinger and total variation distances from probability vectors;
2. compute empirical 1-Wasserstein distance from samples;
3. explain why Wasserstein sees small shifts that TV misses;
4. contrast bounded $f$-divergence metrics with earth-mover distance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform, wasserstein_distance

plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Setup complete")

## 1. Hellinger distance

$$H(p,q)=\sqrt{\tfrac12\sum_i(\sqrt{p_i}-\sqrt{q_i})^2}
=\sqrt{1-\sum_i\sqrt{p_iq_i}}.$$
The square-root transform makes it a bounded metric in $[0,1]$, sensitive to
relative differences in low-probability outcomes.

In [ ]:
def hellinger(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    return float(np.sqrt(0.5 * np.sum((np.sqrt(p) - np.sqrt(q)) ** 2)))


print("identical:", hellinger([0.5, 0.5], [0.5, 0.5]))
print("disjoint: ", hellinger([1, 0], [0, 1]))
print("nearby:   ", round(hellinger([0.9, 0.1], [0.8, 0.2]), 4))

## 2. Wasserstein distance

$W_1$ is the minimum cost of moving probability mass, with cost $|x-y|$.
On the real line it equals the area between cumulative distribution functions.
With samples, `scipy.stats.wasserstein_distance` uses the sorted empirical
quantiles.

In [ ]:
print("delta_0 vs delta_3:", wasserstein_distance([0.0], [3.0]))
print("delta_0 vs delta_1e-6:", wasserstein_distance([0.0], [1e-6]))

rng = np.random.default_rng(0)
x = rng.uniform(0, 1, 20_000)
y = rng.uniform(0, 2, 20_000)
print("U(0,1) vs U(0,2):", round(wasserstein_distance(x, y), 4))

In [ ]:
grid = np.linspace(-0.5, 2.5, 500)
f1, f2 = uniform.cdf(grid, 0, 1), uniform.cdf(grid, 0, 2)
area = np.trapezoid(np.abs(f1 - f2), grid) if hasattr(np, "trapezoid") else \
    np.trapz(np.abs(f1 - f2), grid)

fig, ax = plt.subplots()
ax.plot(grid, f1, label="CDF U(0,1)")
ax.plot(grid, f2, label="CDF U(0,2)")
ax.fill_between(grid, f1, f2, alpha=0.25, color="tab:orange")
ax.set(xlabel="x", ylabel="F(x)", title="W1 is the shaded area between CDFs")
ax.legend()
plt.show()
print(f"numerical area = {area:.4f} (expected 0.5)")

## 3. Total variation distance

$$\mathrm{TV}(p,q)=\tfrac12\sum_i|p_i-q_i|
=\sup_A|p(A)-q(A)|.$$
It is the best possible difference in the probability of any event, so it is
naturally interpreted as a separation test.

In [ ]:
def total_variation(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    return float(0.5 * np.sum(np.abs(p - q)))


print("identical:", total_variation([0.5, 0.5], [0.5, 0.5]))
print("disjoint: ", total_variation([1, 0], [0, 1]))
print("nearby:   ", total_variation([0.6, 0.4], [0.5, 0.5]))
print("disjoint atoms after a tiny shift: TV = 1")

## 4. Geometry versus support

TV and Hellinger only compare probability mass at matching outcomes. A tiny
location shift of disjoint point masses gives $\mathrm{TV}=1$ and $H=1$, while
$W_1$ equals the shift size and vanishes as the points merge.

In [ ]:
p = np.array([0.6, 0.4])
qs = np.linspace(0.01, 0.99, 300)
hell = [hellinger(p, [q, 1 - q]) for q in qs]
tv = [total_variation(p, [q, 1 - q]) for q in qs]

fig, ax = plt.subplots()
ax.plot(qs, hell, label="Hellinger")
ax.plot(qs, tv, label="Total variation")
ax.axvline(0.6, color="0.5", ls="--", label="q = p")
ax.set(xlabel="q1", ylabel="distance", title="Bounded metrics on Bernoulli laws")
ax.legend()
plt.show()

for eps in [1.0, 0.1, 1e-6]:
    print(f"shift {eps:g}: W1={wasserstein_distance([0.0], [eps]):g}, "
          f"TV=1, Hellinger=1")

## Exercises

1. Prove $H^2=1-\mathrm{BC}$ where $\mathrm{BC}=\sum_i\sqrt{p_iq_i}$.
2. Verify that $\mathrm{TV}=\sup_A|p(A)-q(A)|$ for a three-category example.
3. For $N(0,1)$ and $N(\mu,1)$, compute $W_1$ analytically and check by sampling.
4. Find an example where Hellinger and TV rank pairs of distributions differently.
5. Use `Code/information_theory.py` to cross-check the helper implementations.